In [ ]:
import requests
import pandas as pd

CHART_KEY = '56b6f095-2243-4d73-9bcf-57600ef1f38b'

# Khai báo danh sách các trạm muốn lấy (Tên_Trạm: Mã_ID)
STATIONS = {
    'MyThuan': '198f5577f90641f0b885b76070d5fc00',
    'GiangSon': '43308db8bbc24622b8f55426c8c88f3f', 
    'DucXuyen':  'a2e257089eaa49b18f757a9bed24af3e',
    'ChoLach' : 'badd37e15f0d4fa38ae6e0575e8dad5c',
    
    'TraVinh': '2188c6b36fd74225baef4d5ff9ba6487',
    'ChauDoc': 'b9ac7c5068b54fa6b5ad5b59383467a6', 
    'CanTho':  '5c74b82aeece4e8194f06dc61199b9a5',
    'DaiNgai' : '567cc5b1e2de42b2901a7793f718fc75',
    
    'ViThanh': 'b347eda9fd2f40f090ae5a557cab8478',
    'PhungHiep': '5a3bfc6d1a654c0082aada1f5ac242a0', 
}

def crawl_station_data(station_name, station_id, chart_key):
    print(f"Đang xử lý trạm: {station_name}...")
    
    # URL động tự chèn mã trạm vào
    url = f'https://timeseries.api.mrcmekong.org/api/v1/ts/highcharts/{station_id}?sd=2018-04-25T00:00:00.000Z&ed=2026-03-11T00:00:00.000Z'
    
    headers = {
        'Accept': 'application/json, text/plain, */*',
        'Origin': 'https://portal.mrcmekong.org',
        'Referer': 'https://portal.mrcmekong.org/',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36',
        'x-mrc-data-portal-chart-key': chart_key,
    }

    try:
        response = requests.get(url, headers=headers, timeout=60)
        
        if response.status_code == 200:
            data_dict = response.json() 
            
            time_series = data_dict['series'][0]['data']
            
            # Nếu trạm hoàn toàn không có data, bỏ qua để khỏi lỗi
            if not time_series:
                print(f"Trạm {station_name} không có điểm dữ liệu nào. Bỏ qua!\n")
                return
                
            df = pd.DataFrame(time_series, columns=['Timestamp_ms', 'WaterLevel'])
            
            df['ObservationDate'] = pd.to_datetime(df['Timestamp_ms'], unit='ms') + pd.Timedelta(hours=7)
            df = df[['ObservationDate', 'WaterLevel']]
            
            start_year = df['ObservationDate'].dt.year.min()
            end_year = df['ObservationDate'].dt.year.max()
            
            # Gắn biến start_year và end_year vào tên file
            file_name = f"../data/dataCrawl/WaterLevel/{station_name}_WaterLevel_{start_year}_{end_year}_15min.csv"
            
            df.to_csv(file_name, index=False)
            
            print(f"Xong {station_name}: {len(df)} bản ghi.")
            print(f"Dữ liệu thực tế kéo dài từ {start_year} đến {end_year}.")
            print(f"Lưu tại -> {file_name}\n")
            
        else:
            print(f"Lỗi {response.status_code} ở trạm {station_name}\n")
            
    except Exception as e:
        print(f"Lỗi khi tải trạm {station_name} (Có thể JSON rỗng hoặc sai ID): {e}\n")

for name, uid in STATIONS.items():
    crawl_station_data(name, uid, CHART_KEY)

print("TẤT CẢ ĐÃ HOÀN TẤT")